In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 1: Install & Import
# ============================================================
!pip install -q opencv-python-headless scikit-image

import os
import cv2
import json
import numpy as np
from glob import glob
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from skimage.feature import local_binary_pattern
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

print("Imports OK ✅")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 2: Configuration
# ============================================================
class CFG:
    # ── YOUR KAGGLE INPUT PATHS ──
    FF_ROOT    = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
    CELEB_ROOT = "/kaggle/input/datasets/reubensuju/celeb-df-v2"
    GAN_ROOT   = "/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces"   # ← ADJUST

    # ── OUTPUT ──
    OUT        = "/kaggle/working/preprocessed"
    FF_OUT     = "/kaggle/working/preprocessed/ff"
    CELEB_OUT  = "/kaggle/working/preprocessed/celeb"
    GAN_OUT    = "/kaggle/working/preprocessed/gan"
    COMB_OUT   = "/kaggle/working/preprocessed/combined"

    # ── SETTINGS ──
    FRAMES_PER_VIDEO = 10
    FACE_MARGIN      = 0.3
    IMG_SIZE         = 224
    GAN_MAX_PER_CLASS = 5000
    SEED             = 42

cfg = CFG()
np.random.seed(cfg.SEED)

for base in [cfg.FF_OUT, cfg.CELEB_OUT, cfg.GAN_OUT, cfg.COMB_OUT]:
    for sub in ['rgb', 'fft', 'hist']:
        os.makedirs(f"{base}/{sub}", exist_ok=True)

print("Config ready ✅")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 3: Explore Dataset Structure
# ============================================================
def print_tree(path, prefix="", depth=0, max_depth=4, max_files=5):
    if depth >= max_depth or not os.path.exists(path):
        return
    try:
        entries = sorted(os.listdir(path))
    except PermissionError:
        return
    dirs  = [e for e in entries if os.path.isdir(f"{path}/{e}")]
    files = [e for e in entries if os.path.isfile(f"{path}/{e}")]
    for f in files[:max_files]:
        print(f"{prefix}├── {f}")
    if len(files) > max_files:
        print(f"{prefix}├── ... +{len(files)-max_files} more")
    for d in dirs:
        print(f"{prefix}├── {d}/")
        print_tree(f"{path}/{d}", prefix+"│   ", depth+1, max_depth, max_files)

for name, path in [("FF++ c23", cfg.FF_ROOT), ("Celeb-DF", cfg.CELEB_ROOT), ("140K GAN", cfg.GAN_ROOT)]:
    print(f"\n{'='*50}\n{name}: {path}\n{'='*50}")
    if os.path.exists(path):
        print_tree(path)
    else:
        print(f"  ❌ NOT FOUND — adjust CFG path")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 4: Core Functions (Fixed Download)
# ============================================================
import urllib.request
import os

# Try multiple sources for OpenCV DNN model
MODEL_SOURCES = [
    "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/dnn/face_detector/",
    "https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/",
    "https://github.com/opencv/opencv/raw/master/samples/dnn/face_detector/",
]

def download_model_files():
    """Try to download from multiple sources"""
    prototxt_path = "/kaggle/working/deploy.prototxt"
    caffemodel_path = "/kaggle/working/res10_300x300_ssd_iter_140000.caffemodel"
    
    # Check if already downloaded
    if os.path.exists(prototxt_path) and os.path.exists(caffemodel_path):
        print("Model files already exist ✅")
        return True
    
    print("Downloading OpenCV DNN SSD model...")
    
    for base_url in MODEL_SOURCES:
        try:
            print(f"  Trying: {base_url}")
            urllib.request.urlretrieve(
                base_url + "deploy.prototxt", 
                prototxt_path
            )
            urllib.request.urlretrieve(
                base_url + "res10_300x300_ssd_iter_140000.caffemodel", 
                caffemodel_path
            )
            print("  ✅ Download successful!")
            return True
        except Exception as e:
            print(f"  ❌ Failed: {str(e)[:50]}")
            # Clean up partial downloads
            for f in [prototxt_path, caffemodel_path]:
                if os.path.exists(f):
                    os.remove(f)
    
    return False

# Try to download DNN model
dnn_success = download_model_files()

if dnn_success:
    # Use OpenCV DNN (more accurate)
    face_net = cv2.dnn.readNetFromCaffe(
        "/kaggle/working/deploy.prototxt",
        "/kaggle/working/res10_300x300_ssd_iter_140000.caffemodel"
    )
    
    def extract_face(frame, margin=0.3, size=224):
        """Use OpenCV DNN SSD for face detection"""
        h, w = frame.shape[:2]
        
        # Create blob and run forward pass
        blob = cv2.dnn.blobFromImage(
            cv2.resize(frame, (300, 300)), 
            1.0, (300, 300), 
            (104.0, 177.0, 123.0)
        )
        face_net.setInput(blob)
        detections = face_net.forward()
        
        # Find the largest face
        if len(detections[0][0]) == 0:
            # No face detected - center crop
            s = min(h, w)
            y1, x1 = (h-s)//2, (w-s)//2
            crop = frame[y1:y1+s, x1:x1+s]
            return cv2.resize(crop, (size, size))
        
        # Get best detection
        confidences = []
        boxes = []
        for i in range(detections.shape[2]):
            confidence = detections[0][0][i][2]
            if confidence > 0.5:
                box = detections[0][0][i][3:7] * np.array([w, h, w, h])
                (x1, y1, x2, y2) = box.astype("int")
                confidences.append(confidence)
                boxes.append((x1, y1, x2, y2))
        
        if len(boxes) == 0:
            # Fallback to center crop
            s = min(h, w)
            y1, x1 = (h-s)//2, (w-s)//2
            crop = frame[y1:y1+s, x1:x1+s]
            return cv2.resize(crop, (size, size))
        
        # Select largest face by area
        idx = np.argmax([(x2-x1)*(y2-y1) for (x1,y1,x2,y2) in boxes])
        x1, y1, x2, y2 = boxes[idx]
        
        # Apply margin
        fh, fw = frame.shape[:2]
        face_w = x2 - x1
        face_h = y2 - y1
        x1_new = max(0, int(x1 - face_w * margin))
        y1_new = max(0, int(y1 - face_h * margin))
        x2_new = min(fw, int(x2 + face_w * margin))
        y2_new = min(fh, int(y2 + face_h * margin))
        
        crop = frame[y1_new:y2_new, x1_new:x2_new]
        if crop.size == 0:
            return cv2.resize(frame, (size, size))
        return cv2.resize(crop, (size, size))
    
    print("Using OpenCV DNN SSD ✅")
else:
    # Fallback to Haar Cascade (less accurate but works)
    print("\n⚠️  Using Haar Cascade (DNN download failed)")
    print("    Face detection may be less accurate\n")
    
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    )
    
    def extract_face(frame, margin=0.3, size=224):
        """Use Haar Cascade for face detection"""
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(50,50))
        
        if len(faces) == 0:
            h, w = frame.shape[:2]
            s = min(h, w)
            y1, x1 = (h-s)//2, (w-s)//2
            crop = frame[y1:y1+s, x1:x1+s]
            return cv2.resize(crop, (size, size))
        
        areas = [w_ * h_ for (x_, y_, w_, h_) in faces]
        x, y, w_, h_ = faces[np.argmax(areas)]
        
        fh, fw = frame.shape[:2]
        x1 = max(0, int(x - w_*margin))
        y1 = max(0, int(y - h_*margin))
        x2 = min(fw, int(x + w_ + w_*margin))
        y2 = min(fh, int(y + h_ + h_*margin))
        
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            return cv2.resize(frame, (size, size))
        return cv2.resize(crop, (size, size))

# Rest of the functions (unchanged)
def compute_fft(rgb): 
    channels = []
    for c in range(3):
        ch = rgb[:,:,c].astype(np.float32)
        mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(ch))))
        mn, mx = mag.min(), mag.max()
        mag = (mag - mn) / (mx - mn + 1e-8)
        channels.append(mag)
    fft = np.stack(channels, axis=-1)
    return (fft * 255).clip(0,255).astype(np.uint8)

def compute_hist(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    h = cv2.calcHist([hsv],[0],None,[64],[0,180]).flatten()
    s = cv2.calcHist([hsv],[1],None,[32],[0,256]).flatten()
    v = cv2.calcHist([hsv],[2],None,[32],[0,256]).flatten()
    h /= (h.sum()+1e-8); s /= (s.sum()+1e-8); v /= (v.sum()+1e-8)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, 8, 1, 'uniform')
    lbp_h, _ = np.histogram(lbp.ravel(), bins=59, range=(0,59), density=True)
    hsv_f = hsv.astype(np.float32).reshape(-1,3)
    return np.concatenate([
        h, s, v, lbp_h,
        np.mean(hsv_f,0), np.std(hsv_f,0),
        skew(hsv_f,0), kurtosis(hsv_f,0)
    ]).astype(np.float32)

def sample_video_frames(path, n=8):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return []
    idxs = np.linspace(0, total-1, min(n, total), dtype=int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, f = cap.read()
        if ret:
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def save_sample(face, sample_id, out_dir):
    fft = compute_fft(face)
    hist = compute_hist(face)
    np.save(f"{out_dir}/rgb/{sample_id}.npy", face)
    np.save(f"{out_dir}/fft/{sample_id}.npy", fft)
    np.save(f"{out_dir}/hist/{sample_id}.npy", hist)

print("Functions ready ✅")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 5: Discover FF++ Videos
# ============================================================
def find_videos(root, exts=['.mp4','.avi']):
    vids = []
    for ext in exts:
        vids.extend(glob(f"{root}/**/*{ext}", recursive=True))
    return sorted(vids)

def discover_ff(root):
    videos = []
    real_kw = ['original','real','youtube']
    fake_methods = {
        'Deepfakes':      ['Deepfakes','deepfakes','DF'],
        'Face2Face':      ['Face2Face','face2face','F2F'],
        'FaceSwap':       ['FaceSwap','faceswap','FS'],
        'NeuralTextures': ['NeuralTextures','neural_textures','NT'],
    }
    # Find real
    for dp, dn, fn in os.walk(root):
        base = os.path.basename(dp).lower()
        if any(k in base for k in real_kw) and not any(
            k in base for k in ['deepfake','face2face','faceswap','neural']):
            vids = find_videos(dp)
            if vids:
                for v in vids:
                    videos.append({'path':v,'label':0,'method':'real','name':Path(v).stem})
                print(f"  ✅ Real: {len(vids)} in {dp}")
                break
    # Find fakes
    for method, aliases in fake_methods.items():
        found = False
        for dp, dn, fn in os.walk(root):
            base = os.path.basename(dp).lower()
            if any(a.lower() in base for a in aliases):
                vids = find_videos(dp)
                if vids:
                    for v in vids:
                        videos.append({'path':v,'label':1,'method':method,'name':Path(v).stem})
                    print(f"  ✅ {method}: {len(vids)} in {dp}")
                    found = True
                    break
        if not found:
            print(f"  ⚠️  {method} not found")
    return videos

# Also handle pre-extracted frames (images instead of videos)
def discover_ff_images(root):
    videos = []
    for dp, dn, fn in os.walk(root):
        imgs = [f for f in fn if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if len(imgs) < 5:
            continue
        base = dp.lower()
        if any(k in base for k in ['original','real','youtube']) and not any(
            k in base for k in ['deepfake','face2face','faceswap','neural']):
            for img in imgs:
                videos.append({'path':f"{dp}/{img}",'label':0,'method':'real',
                              'name':Path(img).stem,'is_image':True})
        for method, aliases in {'Deepfakes':['deepfake','df'],'Face2Face':['face2face','f2f'],
                                 'FaceSwap':['faceswap','fs'],'NeuralTextures':['neural','nt']}.items():
            if any(a in os.path.basename(dp).lower() for a in aliases):
                for img in imgs:
                    videos.append({'path':f"{dp}/{img}",'label':1,'method':method,
                                  'name':Path(img).stem,'is_image':True})
    return videos

print("Discovering FF++ ...")
ff_videos = discover_ff(cfg.FF_ROOT)
if len(ff_videos) == 0:
    print("  No videos found, trying image mode...")
    ff_videos = discover_ff_images(cfg.FF_ROOT)

counts = defaultdict(int)
for v in ff_videos:
    counts[v['method']] += 1
print(f"\nFF++ Total: {len(ff_videos)}")
for m,c in sorted(counts.items()):
    print(f"  {m}: {c}")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 6: Discover Celeb-DF
# ============================================================
def discover_celeb(root):
    videos = []
    for dp, dn, fn in os.walk(root):
        base = os.path.basename(dp).lower()
        vids = find_videos(dp)
        imgs = [f for f in fn if f.lower().endswith(('.jpg','.jpeg','.png'))]
        items = vids if vids else [f"{dp}/{i}" for i in imgs]
        if not items:
            continue
        is_img = len(vids) == 0
        if any(k in base for k in ['real','youtube','original']):
            for v in items:
                videos.append({'path':v,'label':0,'method':'real',
                              'name':Path(v).stem,'is_image':is_img})
            print(f"  ✅ Real: {len(items)} in {dp}")
        elif any(k in base for k in ['synthesis','fake','swap','manipulated']):
            for v in items:
                videos.append({'path':v,'label':1,'method':'CelebDF',
                              'name':Path(v).stem,'is_image':is_img})
            print(f"  ✅ Fake: {len(items)} in {dp}")
    return videos

print("Discovering Celeb-DF ...")
celeb_videos = discover_celeb(cfg.CELEB_ROOT)
counts = defaultdict(int)
for v in celeb_videos:
    counts[v['method']] += 1
print(f"Celeb-DF Total: {len(celeb_videos)}")
for m,c in sorted(counts.items()):
    print(f"  {m}: {c}")


In [ ]:
# ============================================================
# REPLACE CELL 7 WITH THIS — HANDLES ALL CASES
# ============================================================

def discover_gan_v2(root):
    """
    Bulletproof 140K dataset discovery.
    Handles ANY folder structure.
    """
    images = {'real': [], 'fake': []}
    unknown_dirs = []
    
    if not os.path.exists(root):
        print(f"❌ Path does not exist: {root}")
        print(f"\nAvailable datasets:")
        for d in sorted(os.listdir("/kaggle/input")):
            print(f"  /kaggle/input/{d}")
        return images
    
    print(f"Scanning: {root}")
    print("-" * 60)
    
    for dp, dn, fn in os.walk(root):
        imgs = [f for f in fn if f.lower().endswith(
            ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff')
        )]
        
        if len(imgs) == 0:
            continue
        
        # Build search string from FULL path (not just basename)
        full_path_lower = dp.lower()
        base_lower = os.path.basename(dp).lower()
        parent_lower = os.path.basename(os.path.dirname(dp)).lower()
        
        # ── Check for REAL ──
        real_match = False
        
        # Direct basename matches
        if base_lower in ['real', 'real_faces', 'genuine', 'authentic', 
                          'original', 'training_real', 'validation_real',
                          'test_real', 'real_images', '0']:
            real_match = True
        
        # Partial basename matches
        if not real_match:
            for kw in ['real', 'genuine', 'authentic', 'original']:
                if kw in base_lower and 'fake' not in base_lower:
                    real_match = True
                    break
        
        # Path contains /real/ somewhere
        if not real_match:
            if '/real/' in full_path_lower or '/real' == full_path_lower[-5:]:
                if 'fake' not in base_lower:
                    real_match = True
        
        # Parent folder check (for train/real, valid/real structure)
        if not real_match and base_lower == 'real':
            real_match = True
        
        # ── Check for FAKE ──
        fake_match = False
        
        # Direct basename matches
        if base_lower in ['fake', 'fake_faces', 'synthetic', 'generated',
                          'gan', 'deepfake', 'training_fake', 'validation_fake',
                          'test_fake', 'fake_images', 'spoof', '1']:
            fake_match = True
        
        # Partial basename matches
        if not fake_match:
            for kw in ['fake', 'synthetic', 'generated', 'gan', 'spoof',
                       'manipulated', 'deepfake', 'forged']:
                if kw in base_lower:
                    fake_match = True
                    break
        
        # Path contains /fake/ somewhere
        if not fake_match:
            if '/fake/' in full_path_lower or '/fake' == full_path_lower[-5:]:
                fake_match = True
        
        # ── Resolve conflicts ──
        if real_match and fake_match:
            # "fake" keyword is stronger than "real"
            if any(kw in base_lower for kw in ['fake','synthetic','generated','gan']):
                real_match = False
            else:
                fake_match = False
        
        # ── Assign ──
        full_paths = [os.path.join(dp, f) for f in imgs]
        
        if real_match:
            images['real'].extend(full_paths)
            print(f"  ✅ REAL : {dp}")
            print(f"         {len(imgs)} images")
        elif fake_match:
            images['fake'].extend(full_paths)
            print(f"  ✅ FAKE : {dp}")
            print(f"         {len(imgs)} images")
        else:
            unknown_dirs.append((dp, len(imgs)))
            if len(unknown_dirs) <= 5:
                print(f"  ❓ ???  : {dp}")
                print(f"         {len(imgs)} images, basename='{base_lower}'")
    
    # ── Summary ──
    print(f"\n{'='*60}")
    print(f"RESULT: Real={len(images['real'])}, Fake={len(images['fake'])}")
    
    if images['real']:
        print(f"  Sample real: {images['real'][0]}")
    if images['fake']:
        print(f"  Sample fake: {images['fake'][0]}")
    
    # ── If STILL nothing found ──
    if len(images['real']) == 0 and len(images['fake']) == 0:
        print(f"\n⚠️  AUTO-DETECTION FAILED")
        print(f"\nAll directories with images:")
        
        all_img_dirs = []
        for dp, dn, fn in os.walk(root):
            imgs = [f for f in fn if f.lower().endswith(('.jpg','.jpeg','.png'))]
            if imgs:
                all_img_dirs.append((dp, len(imgs), imgs[0]))
        
        all_img_dirs.sort(key=lambda x: -x[1])
        
        for dp, count, sample in all_img_dirs[:10]:
            print(f"\n  📁 {dp}")
            print(f"     {count} images")
            print(f"     sample: {sample}")
        
        if len(all_img_dirs) == 2:
            print(f"\n💡 HINT: Only 2 image directories found.")
            print(f"   Assuming first = real, second = fake")
            print(f"   Set manually below if this is wrong.")
            
            sorted_dirs = sorted(all_img_dirs, key=lambda x: x[0])
            d1, n1, _ = sorted_dirs[0]
            d2, n2, _ = sorted_dirs[1]
            
            # Guess: alphabetically first might be "fake", second "real"
            # Or vice versa — user needs to verify
            print(f"\n   Directory 1: {d1} ({n1} images)")
            print(f"   Directory 2: {d2} ({n2} images)")
            print(f"\n   UNCOMMENT the manual override in the next cell!")
    
    return images


# ── Run discovery ──
print("Discovering 140K GAN dataset ...")
print("="*60)

if os.path.exists(cfg.GAN_ROOT):
    gan_images = discover_gan_v2(cfg.GAN_ROOT)
else:
    print(f"❌ Path not found: {cfg.GAN_ROOT}")
    
    # Try to find it automatically
    print(f"\nSearching all datasets for image collections...")
    best_match = None
    best_count = 0
    
    for d in sorted(os.listdir("/kaggle/input")):
        root = f"/kaggle/input/{d}"
        if not os.path.isdir(root):
            continue
        
        img_count = 0
        for dp, dn, fn in os.walk(root):
            img_count += sum(1 for f in fn if f.lower().endswith(('.jpg','.jpeg','.png')))
        
        if img_count > 1000:
            print(f"  📦 /kaggle/input/{d} → {img_count} images")
            if img_count > best_count:
                best_count = img_count
                best_match = root
    
    if best_match:
        print(f"\n💡 Best match: {best_match} ({best_count} images)")
        print(f"   Trying this path...")
        cfg.GAN_ROOT = best_match
        gan_images = discover_gan_v2(cfg.GAN_ROOT)
    else:
        gan_images = {'real': [], 'fake': []}
        print("No large image datasets found")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 8: FF++ Processing (8 frames)
# ============================================================

FRAMES_PER_VIDEO = 8  # Changed from 3 to 8 for temporal/rPPG pillars

print(f"Processing FF++: {len(ff_videos)} videos × {FRAMES_PER_VIDEO} frames")
print(f"Expected samples: ~{len(ff_videos) * FRAMES_PER_VIDEO}")

import time
t0 = time.time()

ff_meta = []
for vid in tqdm(ff_videos, desc="FF++"):
    is_image = vid.get('is_image', False)
    
    try:
        if is_image:
            img = cv2.imread(vid['path'])
            if img is None:
                continue
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            face = extract_face_dnn(rgb, cfg.FACE_MARGIN, cfg.IMG_SIZE)
            sid = f"{vid['method']}_{vid['name']}"
            
            np.save(f"{cfg.FF_OUT}/rgb/{sid}.npy", face)
            np.save(f"{cfg.FF_OUT}/fft/{sid}.npy", compute_fft(face))
            np.save(f"{cfg.FF_OUT}/hist/{sid}.npy", compute_hist(face))
            
            ff_meta.append({
                'sample_id': sid, 'video': vid['name'],
                'frame_idx': 0, 'label': vid['label'],
                'method': vid['method'], 'source': 'ff++', 'domain': 'faceswap'
            })
        else:
            frames = sample_video_frames(vid['path'], FRAMES_PER_VIDEO)
            for i, frame in enumerate(frames):
                face = extract_face_dnn(frame, cfg.FACE_MARGIN, cfg.IMG_SIZE)
                sid = f"{vid['method']}_{vid['name']}_{i:03d}"
                
                np.save(f"{cfg.FF_OUT}/rgb/{sid}.npy", face)
                np.save(f"{cfg.FF_OUT}/fft/{sid}.npy", compute_fft(face))
                np.save(f"{cfg.FF_OUT}/hist/{sid}.npy", compute_hist(face))
                
                ff_meta.append({
                    'sample_id': sid, 'video': vid['name'],
                    'frame_idx': i, 'label': vid['label'],
                    'method': vid['method'], 'source': 'ff++', 'domain': 'faceswap'
                })
    except Exception as e:
        continue

elapsed = time.time() - t0
print(f"\n✅ FF++ done in {elapsed/60:.1f} minutes")
print(f"   Total samples: {len(ff_meta)}")

mc = defaultdict(int)
for m in ff_meta:
    mc[m['method']] += 1
for k, v in sorted(mc.items()):
    print(f"   {k}: {v}")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 9 REPLACEMENT: FAST CELEB-DF
# ============================================================

print(f"\nProcessing Celeb-DF: {len(celeb_videos)} videos")

t0 = time.time()
celeb_meta = []

for vid in tqdm(celeb_videos, desc="Celeb-DF"):
    try:
        is_image = vid.get('is_image', False)
        
        if is_image:
            img = cv2.imread(vid['path'])
            if img is None:
                continue
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            face = fast_extract_face(rgb)
        else:
            # Only 1 frame per video for cross-dataset test
            frames = fast_sample_frames(vid['path'], 1)
            if not frames:
                continue
            face = fast_extract_face(frames[0])
        
        sid = f"celeb_{vid['method']}_{vid['name']}"
        
        np.save(f"{cfg.CELEB_OUT}/rgb/{sid}.npy", face)
        np.save(f"{cfg.CELEB_OUT}/fft/{sid}.npy", fast_compute_fft(face))
        np.save(f"{cfg.CELEB_OUT}/hist/{sid}.npy", fast_compute_hist(face))
        
        celeb_meta.append({
            'sample_id': sid, 'video': vid['name'],
            'frame_idx': 0, 'label': vid['label'],
            'method': vid['method'], 'source': 'celeb', 'domain': 'celeb'
        })
    except:
        continue

elapsed = time.time() - t0
n_r = sum(1 for m in celeb_meta if m['label']==0)
n_f = sum(1 for m in celeb_meta if m['label']==1)
print(f"\n✅ Celeb-DF done in {elapsed/60:.1f} minutes")
print(f"   Total: {len(celeb_meta)} ({n_r} real + {n_f} fake)")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 10: Process 140K GAN Dataset
# ============================================================
def process_gan(images, out_dir, max_per_class=5000):
    metadata = []
    for label_name, label_val in [('real', 0), ('fake', 1)]:
        paths = images[label_name].copy()
        np.random.shuffle(paths)
        paths = paths[:max_per_class]
        
        for i, img_path in enumerate(tqdm(paths, desc=f"GAN {label_name}")):
            try:
                img = cv2.imread(img_path)
                if img is None: continue
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                face = cv2.resize(img_rgb, (cfg.IMG_SIZE, cfg.IMG_SIZE))
                
                # Try face detection, fallback to resize
                try:
                    detected = extract_face(img_rgb, cfg.FACE_MARGIN, cfg.IMG_SIZE)
                    if detected is not None:
                        face = detected
                except:
                    pass
                
                method = 'real' if label_val == 0 else 'GAN'
                sid = f"gan_{label_name}_{i:06d}"
                save_sample(face, sid, out_dir)
                metadata.append({
                    'sample_id': sid, 'video': sid,
                    'frame_idx': 0, 'label': label_val,
                    'method': method, 'source': 'gan140k', 'domain': 'gan'
                })
            except:
                pass
    
    n_r = sum(1 for m in metadata if m['label']==0)
    n_f = sum(1 for m in metadata if m['label']==1)
    print(f"GAN processed: {len(metadata)} ({n_r} real + {n_f} fake)")
    return metadata

if len(gan_images['real']) > 0 or len(gan_images['fake']) > 0:
    gan_meta = process_gan(gan_images, cfg.GAN_OUT, cfg.GAN_MAX_PER_CLASS)
else:
    gan_meta = []
    print("Skipping GAN dataset (not found)")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 11: Create Train/Val/Test Splits
# ============================================================
def split_by_video(metadata, train=0.8, val=0.1, seed=42):
    np.random.seed(seed)
    vid_to_samples = defaultdict(list)
    vid_to_method  = {}
    for s in metadata:
        vid_to_samples[s['video']].append(s)
        vid_to_method[s['video']] = s['method']
    
    train_s, val_s, test_s = [], [], []
    
    for method in set(vid_to_method.values()):
        vids = [v for v,m in vid_to_method.items() if m == method]
        np.random.shuffle(vids)
        n = len(vids)
        nt = int(n * train)
        nv = int(n * val)
        for v in vids[:nt]:
            for s in vid_to_samples[v]: s['split'] = 'train'
            train_s.extend(vid_to_samples[v])
        for v in vids[nt:nt+nv]:
            for s in vid_to_samples[v]: s['split'] = 'val'
            val_s.extend(vid_to_samples[v])
        for v in vids[nt+nv:]:
            for s in vid_to_samples[v]: s['split'] = 'test'
            test_s.extend(vid_to_samples[v])
    
    return train_s, val_s, test_s

# Split FF++
ff_train, ff_val, ff_test = split_by_video(ff_meta)
print(f"FF++ — Train:{len(ff_train)} Val:{len(ff_val)} Test:{len(ff_test)}")

# Split GAN
if gan_meta:
    gan_train, gan_val, gan_test = split_by_video(gan_meta)
    print(f"GAN  — Train:{len(gan_train)} Val:{len(gan_val)} Test:{len(gan_test)}")
else:
    gan_train, gan_val, gan_test = [], [], []

# Combined
comb_train = ff_train + gan_train
comb_val   = ff_val   + gan_val
comb_test  = ff_test  + gan_test
np.random.shuffle(comb_train)

print(f"\nCombined — Train:{len(comb_train)} Val:{len(comb_val)} Test:{len(comb_test)}")

# Celeb-DF is ALL test (cross-dataset)
for s in celeb_meta:
    s['split'] = 'test'

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 12: Save All Metadata
# ============================================================
# FF++ splits
for name, data in [('train',ff_train),('val',ff_val),('test',ff_test)]:
    with open(f"{cfg.FF_OUT}/metadata_{name}.json",'w') as f:
        json.dump(data, f)
    print(f"FF++ {name}: {len(data)}")

# GAN splits
if gan_meta:
    for name, data in [('train',gan_train),('val',gan_val),('test',gan_test)]:
        with open(f"{cfg.GAN_OUT}/metadata_{name}.json",'w') as f:
            json.dump(data, f)
        print(f"GAN {name}: {len(data)}")

# Combined splits
for name, data in [('train',comb_train),('val',comb_val),('test',comb_test)]:
    with open(f"{cfg.COMB_OUT}/metadata_{name}.json",'w') as f:
        json.dump(data, f)
    print(f"Combined {name}: {len(data)}")

# Celeb-DF
with open(f"{cfg.CELEB_OUT}/metadata_test.json",'w') as f:
    json.dump(celeb_meta, f)
print(f"Celeb-DF test: {len(celeb_meta)}")

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 13: Compute Histogram Stats
# ============================================================
def compute_stats(train_meta, ff_dir, gan_dir, save_dir):
    hists = []
    for s in tqdm(train_meta, desc="Hist stats"):
        src = s.get('source','ff++')
        d = gan_dir if src == 'gan140k' else ff_dir
        p = f"{d}/hist/{s['sample_id']}.npy"
        if os.path.exists(p):
            hists.append(np.load(p))
    hists = np.stack(hists)
    mean = hists.mean(0)
    std  = hists.std(0)
    std[std < 1e-8] = 1.0
    np.save(f"{save_dir}/hist_mean.npy", mean)
    np.save(f"{save_dir}/hist_std.npy", std)
    print(f"Hist stats saved — dim:{mean.shape[0]}")
    return mean, std

hist_mean, hist_std = compute_stats(comb_train, cfg.FF_OUT, cfg.GAN_OUT, cfg.COMB_OUT)

In [ ]:
# ============================================================
# NOTEBOOK 1 — CELL 14: Verify & Disk Usage
# ============================================================
for name, d in [("FF++",cfg.FF_OUT),("GAN",cfg.GAN_OUT),("Celeb",cfg.CELEB_OUT)]:
    n = len(glob(f"{d}/rgb/*.npy"))
    print(f"{name}: {n} samples")

print(f"\nDisk usage:")
os.system(f"du -sh {cfg.OUT}")
os.system(f"du -sh {cfg.FF_OUT}")
os.system(f"du -sh {cfg.GAN_OUT}")
os.system(f"du -sh {cfg.CELEB_OUT}")

print(f"\n{'='*50}")
print("NOTEBOOK 1 COMPLETE ✅")
print("Save output as Kaggle Dataset: 'xdeep-preprocessed'")
print(f"{'='*50}")


In [ ]:
# ==============================================================
# Convert float32 FFT to uint8 (saves 75% space)
# ==============================================================
import numpy as np
from glob import glob
from tqdm import tqdm

def compress_fft_files(base_dir):
    """Convert FFT float32 → uint8 to save disk space."""
    fft_files = glob(f"{base_dir}/fft/*.npy")
    saved = 0
    for f in tqdm(fft_files, desc="Compressing FFT"):
        data = np.load(f)
        if data.dtype == np.float32:
            data_uint8 = (data * 255).clip(0, 255).astype(np.uint8)
            np.save(f, data_uint8)
            saved += 1
    print(f"  Compressed {saved} FFT files")

for name, d in [("FF++", "/kaggle/working/preprocessed/ff"),
                ("GAN", "/kaggle/working/preprocessed/gan"),
                ("Celeb", "/kaggle/working/preprocessed/celeb")]:
    print(f"\n{name}:")
    compress_fft_files(d)

# ==============================================================
# Package into single tar archive for Kaggle upload
# ==============================================================
import subprocess, os

subprocess.run([
    "tar", "-cf", "/kaggle/working/preprocessed.tar",
    "-C", "/kaggle/working", "preprocessed"
], check=True)

# Remove raw folder so only the .tar remains in output
subprocess.run(["rm", "-rf", "/kaggle/working/preprocessed"], check=True)

size_gb = os.path.getsize('/kaggle/working/preprocessed.tar') / 1024**3
print(f"\nArchive: {size_gb:.2f} GB")
print("DONE — Save as New Dataset from notebook Output")

In [ ]:
# ============================================================
# NOTEBOOK 1 — FINAL CELL: Cleanup & Package
# ============================================================
import os
import shutil
import subprocess

print("="*60)
print("CLEANUP: Removing model files and cache...")
print("="*60)

# Delete DNN model files
for f in ['deploy.prototxt', 'res10_300x300_ssd_iter_140000.caffemodel']:
    path = f"/kaggle/working/{f}"
    if os.path.exists(path):
        os.remove(path)
        print(f"  ✅ Deleted {f}")

# Delete __pycache__ and .pyc files
for root, dirs, files in os.walk("/kaggle/working"):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))
    for f in files:
        if f.endswith('.pyc'):
            os.remove(os.path.join(root, f))

# Verify what remains
print("\n📁 Remaining files in /kaggle/working/:")
os.system("du -sh /kaggle/working/* 2>/dev/null | sort -hr")

# Create tar archive
print("\n📦 Creating archive...")
subprocess.run([
    "tar", "-czf", "/kaggle/working/preprocessed.tar.gz",
    "-C", "/kaggle/working", "preprocessed"
], check=True)

final_size = os.path.getsize('/kaggle/working/preprocessed.tar.gz') / 1024**3
print(f"\n✅ Archive size: {final_size:.2f} GB")
print("✅ Ready to save as Kaggle dataset!")